In [1]:
from winogender_contextuality.utils import *
from winogender_contextuality.modeling.contextuality import *
from winogender_contextuality.config import * 
from winogender_contextuality.modeling.prompting import *
import ast
#from winogender_contextuality.modeling.meta_prompting import *
import matplotlib.pyplot as plt
import numpy as np

2026-01-20 16:14:41.712 | INFO     | winogender_contextuality.config:<module>:13 - PROJ_ROOT path is: /home/sagar/winogender_contextuality
2026-01-20 16:14:41.713 | INFO     | winogender_contextuality.config:<module>:17 - DATA_ROOT path is: /data_users1/sagar/winogender_contextuality


# Constants

In [2]:
questions = ['anaphora', 'pos', 'other_gender']

In [3]:
llama8b = load_ndjson(INTERIM_DATA_DIR / "metaprompting_Llama-3.1-8B-Instruct_0.5_1017090126.ndjson")

In [4]:
#gpt03 = load_ndjson(INTERIM_DATA_DIR / "metaprompting_measurements_gpt-oss-20b_0.3.ndjson")
gpt = load_ndjson(INTERIM_DATA_DIR / "metaprompting_gpt-oss-20b_0.5_2321311225.ndjson")
#gpt07 = load_ndjson(INTERIM_DATA_DIR / "metaprompting_measurements_gpt-oss-20b_0.7.ndjson")

In [5]:
phi = load_ndjson(INTERIM_DATA_DIR / "metaprompting_phi-4_0.5_2330311225.ndjson")

In [6]:
qwen = load_ndjson(INTERIM_DATA_DIR / "metaprompting_Qwen2.5-7B-Instruct_0.5_0013010126.ndjson" )

In [7]:
llama1b = load_ndjson(INTERIM_DATA_DIR / "metaprompting_Llama-3.2-1B-Instruct_0.5_1346081025.ndjson")

In [8]:
gemma = load_ndjson(INTERIM_DATA_DIR / "metaprompting_gemma-3-12b-it_0.5_1754190126.ndjson")

In [14]:
models = [gpt, phi, gemma, llama1b, llama8b, qwen]

In [15]:
model_names = ["gpt", "phi", "gemma", "llama1b", "llama8b", "qwen"]

# Functions

In [9]:
def get_meta_index(idx: int,
                   lst: list[dict]):

    """
    :param idx: pair index
    :param lst: a list of MetaQA objects or equivalent dictionaries 

    Filters list of responses by index.
    """

    return [x for x in lst if x['index'] == idx]
    
def get_meta_question(q: str,
                      lst: list[dict]):

    """
    :param q: string key of the question
    :param lst: a list of MetaQA objects or equivalent dictionaries 

    Filters list of responses by question.
    """

    return [x for x in lst if x['question'] == q]

In [10]:
def get_meta_scores(lst: list[dict],
                   questions: list[str] = ['anaphora', 'pos', 'other_gender']):
    
    dic = {idx: {} for idx in range(60)}


    for idx in range(60):
        for q in questions:
            responses = get_meta_question(q, get_meta_index(idx, lst))
            n_correct = len([r for r in responses if r['response'] == r['answer'].replace("the ", "").replace("a ", "").strip().lower()])
            dic[idx][q] = n_correct/8 # as a fraction of best possible (including errors)

    return dic
        

In [11]:
def get_average_meta(dic: dict):

    response_arr = []
    for idx, dic in dic.items():
        response_arr.append(list(dic.values()))
    response_arr = np.array(response_arr)

    return np.mean(response_arr, axis=0)

In [12]:
def plot_response_dict(response_d):
    # Extract data
    indices = list(response_d.keys())
    categories = list(next(iter(response_d.values())).keys())
    colors = plt.cm.viridis(np.linspace(0, 1, len(categories)))  # color per category

    # Create plot
    plt.figure(figsize=(8, 5))

    for cat, color in zip(categories, colors):
        vals = [response_d[i][cat] for i in indices]
        plt.scatter(indices, vals, color=color, label=cat, s=50, alpha=0.7)

    # Labels and style
    plt.xlabel('Index')
    plt.ylabel('Score')
    plt.title('Response Dictionary Scatter Plot')
    plt.xticks(indices)
    plt.ylim(0, 1)
    plt.legend(title='Category')
    plt.grid(True, linestyle='--', alpha=0.6)

# Counting Errors

In [ ]:
# This is maybe not robust because there are instances where it says something like "BLANK" or something

# Counting Correct Responses

# Metaprompting Scores 

In [16]:
for model, model_name in zip(models, model_names):
    scores = get_meta_scores(model)
    average = get_average_meta(scores)

    print(f"{model_name}: {average}")

gpt: [0.62083333 1.         0.85      ]
phi: [0. 0. 0.]
gemma: [0. 0. 0.]
llama1b: [0.         0.00208333 0.        ]
llama8b: [0.58333333 0.9375     0.96458333]
qwen: [0. 0. 0.]
